In [1]:
import pandas as pd
import numpy as np
from os import listdir
from os.path import isfile, join
import re
import json
import ast
sequences = pd.read_csv("/home/lz80/un-xPass/unxpass/steffen/sequences_new.csv")
sequences['location'] = sequences['location'].apply(lambda x: ast.literal_eval(x))
sequences['statsbomb_x'] = sequences.apply(lambda d: d['location'][0], axis = 1)
sequences['statsbomb_y'] = sequences.apply(lambda d: d['location'][1], axis = 1)
sequences[['statsbomb_x', 'statsbomb_y', 'ball_x', 'ball_y']]
#sequences['location']
games = pd.read_json("/home/lz80/rdf/sp161/shared/soccer-decision-making/womens_euro_receipts/matches/53/106.json", convert_dates = False)#directory of statsbomb matches data
games['home_team'] = games.apply(lambda d: d['home_team']['home_team_name'], axis = 1).str.replace("Women's", "").str.replace("WNT", "").str.strip()
games['away_team'] = games.apply(lambda d: d['away_team']['away_team_name'], axis = 1).str.replace("Women's", "").str.replace("WNT", "").str.strip()
game_dir = "/home/lz80/rdf/sp161/shared/soccer-decision-making/allHawkEye/"#directory of hawkeye data
dirfiles = [f for f in listdir(game_dir) if not f.startswith('.')]
matches_map = {}
for game in dirfiles:
    home_team = game.split('_')[1]
    away_team = game.split('_')[2]
    #print(games['away_team'])
    game_id = games[(games['home_team'] == home_team) & (games['away_team'] == away_team)].reset_index().loc[0]['match_id']
    matches_map[game_id] = game
uefa_map = dict(zip(sequences['uefa_player_id'], sequences['player_id']))
def getGKTeam(game_id):
    lineups = f"/home/lz80/rdf/sp161/shared/soccer-decision-making/Hawkeye_AllGames/lineups/{game_id}.json"
    lineup_df = pd.read_json(lineups, convert_dates = False)
    team_1 = lineup_df['team_id'].loc[0]
    team_2 = lineup_df['team_id'].loc[1]
    team_1_dict = lineup_df['lineup'].loc[0]
    team_2_dict = lineup_df['lineup'].loc[1]
    team_1_lineup = [player_dict['player_id'] for player_dict in team_1_dict]
    team_2_lineup = [player_dict['player_id'] for player_dict in team_2_dict]
    global player_to_team
    team_map = {team_1 : team_1_lineup, team_2 : team_2_lineup}
    player_to_team = {player_id: team_id for team_id, players in team_map.items() for player_id in players}
    pos_dict = {player['player_id']: player['positions'][0]['position'] for player in team_1_dict if len(player['positions']) > 0}
    team_2_pos_dict = {player['player_id']: player['positions'][0]['position'] for player in team_2_dict if len(player['positions']) > 0}
    pos_dict.update(team_2_pos_dict)
    goalkeepers = [key for (key,value) in pos_dict.items() if value == "Goalkeeper"]
    return goalkeepers, player_to_team

def getPlayer(timestamp, match, period, team):
    print(timestamp)
    goalkeepers, player_to_team = getGKTeam(match)
    match = matches_map[match]
    minute = int(timestamp // 60) + 1
    second = timestamp % 60
    player_tracking_dir = f"/home/lz80/rdf/sp161/shared/soccer-decision-making/allHawkEye/{match}/scrubbed.samples.centroids/"
    dirfiles = [f for f in listdir(player_tracking_dir) if not f.startswith('.')]
    sample_file = dirfiles[0]
    file_path_begin = "_".join(sample_file.split('_')[0:3])
    if (period == 1 and minute > 45) or (period == 2 and minute > 90) or (period == 3 and minute > 105) or (period == 4 and minute > 120):
        player_loc_path = f"{player_tracking_dir}{file_path_begin}_{str(period)}_{max_min[period]}_{minute - max_min[period]}.football.samples.centroids"
    else:
        player_loc_path = f"{player_tracking_dir}{file_path_begin}_{str(period)}_{str(minute)}.football.samples.centroids"
    player_df_all = pd.read_json(player_loc_path, lines = True, orient = 'columns')
    player_dict = player_df_all['samples'].loc[0]['people']
    player_df = pd.DataFrame(player_dict)#['centroid'].loc[0][0]
    player_df['pos'] = player_df.apply(lambda d: d['centroid'][0]['pos'], axis = 1)
    player_df['statsbombid'] = player_df.apply(lambda d: d['personId']['uefaId'], axis = 1).astype(int).map(uefa_map)
    player_df['team'] = player_df['statsbombid'].map(player_to_team)
    player_df['teammate'] = np.where(player_df['team'] == team, True, False) 
    player_df['isGK'] = np.where(player_df['statsbombid'].isin(goalkeepers), True, False)#finds location of teammates goalkeeper
    needflip = False
    if player_df[(player_df['isGK']) & (player_df['teammate'])].shape[0] == 0:
        print("No GK Found")
        #missinggks.append([action_id])  
    elif player_df[(player_df['isGK']) & (player_df['teammate'])].reset_index(drop = True).loc[0]['pos'][0] > 0:
        needflip = True
    return needflip
getPlayer(11.207, 3835327, 1, 852)

11.207


True

In [4]:
import pandas as pd
import numpy as np
from os import listdir
from os.path import join
import re
import json
import ast

# Load sequences and convert location column to lists
sequences = pd.read_csv("/home/lz80/un-xPass/unxpass/steffen/sequences_new.csv")
sequences['location'] = sequences['location'].apply(ast.literal_eval)

# Directly extract statsbomb_x and statsbomb_y without using apply()
sequences['statsbomb_x'] = sequences['location'].str[0]
sequences['statsbomb_y'] = sequences['location'].str[1]

# Load match data and clean team names
games = pd.read_json("/home/lz80/rdf/sp161/shared/soccer-decision-making/womens_euro_receipts/matches/53/106.json", convert_dates=False)
games['home_team'] = games['home_team'].apply(lambda d: d['home_team_name'].replace("Women's", "").replace("WNT", "").strip())
games['away_team'] = games['away_team'].apply(lambda d: d['away_team_name'].replace("Women's", "").replace("WNT", "").strip())

# Map files to match IDs
game_dir = "/home/lz80/rdf/sp161/shared/soccer-decision-making/allHawkEye/"
dirfiles = [f for f in listdir(game_dir) if not f.startswith('.')]
matches_map = {
    games.loc[
        (games['home_team'] == f.split('_')[1]) & 
        (games['away_team'] == f.split('_')[2])
    ]['match_id'].values[0]: f
    for f in dirfiles
}

# Map player ids
uefa_map = dict(zip(sequences['uefa_player_id'], sequences['player_id']))

# Get goalkeeper data
def getGKTeam(game_id):
    lineups = f"/home/lz80/rdf/sp161/shared/soccer-decision-making/Hawkeye_AllGames/lineups/{game_id}.json"
    lineup_df = pd.read_json(lineups, convert_dates=False)
    
    # Extract players and positions
    team_1_lineup = [player['player_id'] for player in lineup_df['lineup'].loc[0]]
    team_2_lineup = [player['player_id'] for player in lineup_df['lineup'].loc[1]]
    
    team_map = {lineup_df['team_id'].loc[0]: team_1_lineup, lineup_df['team_id'].loc[1]: team_2_lineup}
    player_to_team = {player_id: team_id for team_id, players in team_map.items() for player_id in players}

    pos_dict = {
        player['player_id']: player['positions'][0]['position'] 
        for lineup in [lineup_df['lineup'].loc[0], lineup_df['lineup'].loc[1]] 
        for player in lineup if len(player['positions']) > 0
    }
    
    # Get goalkeepers
    goalkeepers = [key for key, value in pos_dict.items() if value == "Goalkeeper"]
    return goalkeepers, player_to_team

# Get player data
def getPlayer(timestamp, match, period, team):
    goalkeepers, player_to_team = getGKTeam(match)
    match_file = matches_map[match]
    
    # Calculate minute and second
    minute = int(timestamp // 60) + 1
    player_tracking_dir = f"/home/lz80/rdf/sp161/shared/soccer-decision-making/allHawkEye/{match_file}/scrubbed.samples.centroids/"
    sample_file = [f for f in listdir(player_tracking_dir) if not f.startswith('.')][0]
    file_path_begin = "_".join(sample_file.split('_')[0:3])

    # Choose the appropriate player location file
    player_loc_path = f"{player_tracking_dir}{file_path_begin}_{str(period)}_{str(minute)}.football.samples.centroids"
    player_df_all = pd.read_json(player_loc_path, lines=True, orient='columns')
    
    # Process player locations
    player_df = pd.DataFrame(player_df_all['samples'].loc[0]['people'])
    player_df['pos'] = player_df['centroid'].apply(lambda d: d[0]['pos'])
    player_df['statsbombid'] = player_df['personId'].apply(lambda d: int(d['uefaId'])).map(uefa_map)
    player_df['team'] = player_df['statsbombid'].map(player_to_team)
    player_df['teammate'] = player_df['team'] == team
    player_df['isGK'] = player_df['statsbombid'].isin(goalkeepers)

    needflip = player_df[(player_df['isGK']) & (player_df['teammate'])]['pos'].str[0].gt(0).any()
    
    return needflip

# Example usage
getPlayer(11.207, 3835327, 1, 852)


True

In [5]:
sequences = sequences.dropna(subset = 'BallReceipt')

In [9]:
new_sequences = sequences.head(50)

In [10]:
new_sequences['NeedsFlip'] = new_sequences.apply(lambda d: getPlayer(d['BallReceipt'], d['match_id'], d['period'], d['team_id']), axis = 1)


/tmp/ipykernel_11730/3345098701.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_sequences['NeedsFlip'] = new_sequences.apply(lambda d: getPlayer(d['BallReceipt'], d['match_id'], d['period'], d['team_id']), axis = 1)


In [14]:
new_sequences['RightSide'] = np.where(new_sequences["ball_x"] > 0, 1, 0)
new_sequences['NeedsFlip'] = np.where(new_sequences['NeedsFlip'], 1, 0)


/tmp/ipykernel_11730/3150830178.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_sequences['RightSide'] = np.where(new_sequences["ball_x"] > 0, 1, 0)
/tmp/ipykernel_11730/3150830178.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_sequences['NeedsFlip'] = np.where(new_sequences['NeedsFlip'], 1, 0)


,ball_x,ball_y,statsbomb_x,statsbomb_y,RightSide,NeedsFlip
0,38.473638,2.227355,15.5,43.4,1,1
1,26.744944,19.962049,26.3,62.9,1,1
2,34.546724,-12.716576,18.5,28.9,1,1
3,-1.518847,20.134390,65.2,62.9,0,1
4,-16.893727,-10.731638,77.4,30.2,0,1
5,-14.905724,12.341628,77.0,58.1,0,1
6,31.441994,10.474667,21.1,53.6,1,1
7,23.669441,-14.882010,32.5,29.4,1,1
8,-1.162478,-20.841884,65.0,20.5,0,1
9,10.959964,-28.214479,70.8,72.3,1,0


In [15]:
test = new_sequences[['ball_x','ball_y', 'statsbomb_x', 'statsbomb_y', 'RightSide', 'NeedsFlip']]